In [1]:
import pandas as pd
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

W1104 12:53:00.940000 5754 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## BERTopic

### Setup

In [14]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english")
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=embedding_model,          # Extract embeddings
  umap_model=umap_model,                    # Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Tokenize topics
  ctfidf_model=ctfidf_model,                # Extract topic words
  representation_model=representation_model # Fine-tune topic representations
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
df = pd.read_excel("../paragrafos_util.xlsx")
docs = df['texto_paragrafo'].tolist()
timestamps = df['created_at'].tolist()

### Training

In [16]:
topics, probs = topic_model.fit_transform(docs)

### Loading

In [ ]:
topic_model = BERTopic.load("", embedding_model="all-MiniLM-L6-v2")
topics, _ = topic_model.transform(docs)
topic_model.topics_ = topics

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Model Evaluation

In [16]:
import numpy as np
t = np.array(topic_model.topics_)

print("docs:", len(docs), "| topics_:", len(t))
print("outliers em topics_:", (t == -1).sum())
print("outliers em topic_sizes_:", topic_model.topic_sizes_.get(-1))
print("total em topic_sizes_:", sum(topic_model.topic_sizes_.values()))

docs: 10312 | topics_: 10312
outliers em topics_: 3162
outliers em topic_sizes_: 3162
total em topic_sizes_: 10312


In [10]:
info = topic_model.get_topic_info()
n_topicos = (info.Topic != -1).sum()
pct_outliers = info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum()
print(n_topicos, f"{pct_outliers:.1%}")

reais = info[info.Topic != -1]
print(reais.Count.describe())
print(reais.nlargest(5, "Count")[["Topic", "Count", "Name"]])

69 34.5%
count     69.000000
mean      97.956522
std      120.990086
min       15.000000
25%       28.000000
50%       56.000000
75%      120.000000
max      679.000000
Name: Count, dtype: float64
   Topic  Count                                Name
1      0    679      0_inflation_easing_yields_euro
2      1    578     1_inflation_tariffs_tariff_2026
3      2    376        2_currencies_currency_fx_usd
4      3    358    3_tariffs_tariff_exports_imports
5      4    282  4_cpi_consensus_forecasts_forecast


In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def avaliar(topic_model, docs, top_n=10):
    topics_ = np.array(topic_model.topics_)
    ids = [t for t in topic_model.get_topics() if t != -1]
    # words = [[w for w, _ in topic_model.get_topic(t)[:top_n]] for t in ids]

    # analyzer = topic_model.vectorizer_model.build_analyzer()
    # tokens = [analyzer(d) for d in docs]
    # dic = Dictionary(tokens)
    # words_ok = [[w for w in ws if w in dic.token2id] for ws in words]
    # npmi = CoherenceModel(topics=words_ok, texts=tokens, dictionary=dic,
    #                       coherence="c_npmi").get_coherence()

    # flat = [w for ws in words for w in ws]
    # diversidade = len(set(flat)) / len(flat)

    off = 1 if -1 in topic_model.get_topics() else 0
    emb = np.array(topic_model.topic_embeddings_)[[t + off for t in ids]]
    sim = cosine_similarity(emb); np.fill_diagonal(sim, 0)

    sizes = np.bincount(topics_[topics_ != -1])
    sizes = sizes[sizes > 0]
    return {
        "n_topicos": len(ids),
        "outliers_%": round((topics_ == -1).mean() * 100, 1),
        "mediana_docs": int(np.median(sizes)),
        "viz_mais_proximo_medio": round(sim.max(axis=1).mean(), 3),
    }

antes = avaliar(topic_model, docs)
print(antes, sep="\n")

{'n_topicos': 44, 'outliers_%': np.float64(30.7), 'mediana_docs': 94, 'viz_mais_proximo_medio': np.float32(0.809)}


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

tm = modelo_manual
off = 1 if -1 in tm.get_topics() else 0
ids = sorted(t for t in tm.get_topics() if t != -1)
rows = [t + off for t in ids]

S_emb   = cosine_similarity(np.array(tm.topic_embeddings_)[rows])
S_tfidf = cosine_similarity(tm.c_tf_idf_[rows])
S = (S_emb + S_tfidf) / 2
np.fill_diagonal(S, 0)

# 1) ranking de pares candidatos
pares = [(ids[i], ids[j], S[i, j], S_emb[i, j], S_tfidf[i, j])
         for i in range(len(ids)) for j in range(i + 1, len(ids))]
pares = pd.DataFrame(pares, columns=["a", "b", "score", "emb", "tfidf"]).sort_values("score", ascending=False)
pares["palavras_a"] = pares.a.map(lambda t: ", ".join(w for w, _ in tm.get_topic(t)[:5]))
pares["palavras_b"] = pares.b.map(lambda t: ", ".join(w for w, _ in tm.get_topic(t)[:5]))
pares.head(60).to_csv("candidate_pairs.csv", index=False)

# 2) grupos automáticos (linkage "average" evita encadear tópicos)
D = np.clip(1 - S, 0, None); np.fill_diagonal(D, 0)
ag = AgglomerativeClustering(n_clusters=None, metric="precomputed",
                             linkage="average", distance_threshold=0.45)
lab = ag.fit_predict(D)
grupos = [g for g in pd.Series(ids).groupby(lab).apply(list) if len(g) > 1]
for g in grupos:
    print(g, "|", " / ".join(", ".join(w for w, _ in tm.get_topic(t)[:3]) for t in g))

[0, 3, 23] | dollar, usd, fx / jpy, yen, boj / carry, brl, zar
[69, 118] | home, sales, existing / family, starts, permits
[7, 22, 41, 117] | pce, core, year / pce, ppi, cpi / pce, core, year / import, petroleum, prices
[10, 36, 74] | ai, capex, investment / ai, occupations, workers / adoption, ai, productivity
[25, 34, 102] | inflows, equity, em / funds, fixed, bond / sector, inflows, technology
[8, 16, 17, 66] | powell, restrictive, fed / 25bp, cuts, cut / fomc, meeting, funds / dovish, probability, pricing
[68, 88, 125] | expectations, inflation, measure / michigan, expectations, university / umich, sentiment, expectations
[30, 96] | 75, consensus, line / decisions, bank, central
[15, 56] | tga, funding, qt / coupon, sizes, treasury
[4, 43, 131] | tracking, gdp, quarter / q4, gdp, 2025 / opinative, impacts, tracking
[38, 116] | immigration, immigrants, enforcement / population, household, participation
[79, 86, 128] | canada, japan, area / vs, area, canada / food, japan, energy
[39,

In [20]:
import numpy as np
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer

def npmi_diversidade(topic_model, docs, top_n=10):
    ids = [t for t in topic_model.get_topics() if t != -1]
    palavras = {t: [w for w, _ in topic_model.get_topic(t)[:top_n]] for t in ids}
    vocab = sorted({w for ws in palavras.values() for w in ws})
    idx = {w: i for i, w in enumerate(vocab)}

    cv = CountVectorizer(vocabulary=vocab, ngram_range=(1, 3), binary=True)
    X = cv.transform(docs)                       # documentos x palavras (presença)
    N = X.shape[0]
    p = np.asarray(X.sum(axis=0)).ravel() / N
    co = (X.T @ X).toarray() / N                 # co-ocorrência por documento

    scores = []
    for t in ids:
        pares = []
        for a, b in combinations(palavras[t], 2):
            i, j = idx[a], idx[b]
            if p[i] == 0 or p[j] == 0:
                continue
            pij = co[i, j]
            pares.append(-1.0 if pij == 0 else np.log(pij / (p[i] * p[j])) / -np.log(pij))
        if pares:
            scores.append(np.mean(pares))

    flat = [w for ws in palavras.values() for w in ws]
    return {"npmi": round(float(np.mean(scores)), 3),
            "diversidade": round(len(set(flat)) / len(flat), 3)}

print(npmi_diversidade(topic_model, docs))
print(npmi_diversidade(modelo_manual, docs))

{'npmi': -0.047, 'diversidade': 0.361}
{'npmi': 0.283, 'diversidade': 0.621}


## Visualization

### Topics Over Time

In [6]:
topics_over_time = topic_model.topics_over_time(docs, timestamps)

2026-11-04 12:53:50,596 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 491) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 


In [20]:
topic_model.visualize_topics()

In [8]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.save("workspaces/occam-project/saves/awa", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [7]:
freq = topic_model.get_topic_info()
top_ids = freq[freq.Topic != -1]["Topic"].tolist()

fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=top_ids,        # só os 15 maiores
    normalize_frequency=True,
)

fig

In [10]:
# print(modelo_manual.get_topic(1))
# print(modelo_manual.get_topic(12))
topics = modelo_manual.topics_
[7,22,41]
[11,47]
[1,12]
print([doc for doc, topic in zip(docs, topics) if topic == 4])
print([doc for doc, topic in zip(docs, topics) if topic == 43])

['- terminal rate forecast unchanged at 3–3.25%.', 'net-net a wash for growth: the q2 gdp tracking estimate holds at +1.7% (domestic final sales +1.5%) and the past-quarter estimate at +2.1%.', '- q4 gdp tracking at +2.3% quarter-over-quarter annualized, left unchanged on the week.', '- q1 gdp tracking estimate left unchanged at +1.3%.', '- peak financial conditions drag on us gdp growth now estimated at 0.2pp in q3, down sharply from 1.0pp on april 8.', 'the firmer-than-assumed retail details lifted the q2 gdp tracking estimate by 0.2pp to +2.4% quarter-over-quarter annualized, with domestic final sales at +2.3%.', 'q1 gdp tracking cut by 0.2pp to +2.7% quarter-over-quarter annualized, with the downgrade attributed to softer inventory build.', '- q1 private domestic final sales tracking holds at +2.3%, consistent with the firmer core orders and shipments underneath the headline.', '- q2 gdp tracking estimate raised 0.3pp to 4.1% last week.', 'q1 gdp tracking estimate raised 0.5pp to 3

In [17]:
hierarquical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarquical_topics)

100%|██████████| 121/121 [00:07<00:00, 15.39it/s]


In [7]:
topicos_validos = [t for t in modelo_manual.get_topics().keys() if t != -1]
matriz_sim = modelo_manual.approximate_distribution(docs)[0]

from sklearn.metrics.pairwise import cosine_similarity
embeddings_topicos = modelo_manual.topic_embeddings_[1:]
matriz_sim = cosine_similarity(embeddings_topicos)

df_sim = pd.DataFrame(matriz_sim, index=topicos_validos, columns=topicos_validos)

pares = df_sim.unstack().reset_index()
pares.columns = ['Topico_A', 'Topico_B', 'Similaridade']
pares_filtrados = pares[pares['Topico_A'] < pares['Topico_B']]

top_similares = pares_filtrados.sort_values(by='Similaridade', ascending=False)

print(top_similares.head(20))

       Topico_A  Topico_B  Similaridade
946           7        22      0.902925
2945         22        41      0.875678
965           7        41      0.860792
1499         11        47      0.857723
144           1        12      0.852608
9474         71       102      0.847952
571           4        43      0.847163
10388        78        92      0.847103
4430         33        74      0.843488
10514        79        86      0.842815
8384         63        68      0.840715
6849         51       117      0.837820
1926         14        78      0.837361
1644         12        60      0.836444
2800         21        28      0.833588
1604         12        20      0.830339
9063         68        87      0.828231
23            0        23      0.827744
1133          8        77      0.826475
1481         11        29      0.826313


## Saving Results

In [ ]:
topic_model.save("backups/min 15", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)
topics_over_time.to_csv("resultados/min 15/topics_over_time.csv")
fig.write_html("resultados/min 15/topics_over_time.html")

In [11]:
import plotly.graph_objects as go

# 1) tabela período x tópico, sem outliers
t = topics_over_time[topics_over_time.Topic != -1]
tab = (t.pivot_table(index="Timestamp", columns="Topic",
                     values="Frequency", aggfunc="sum", fill_value=0)
         .sort_index())

tab.index = pd.to_datetime(tab.index)
tab = tab.resample("W").sum()

# 2) participação por período (fluxo)
share = tab.div(tab.sum(axis=1), axis=0)

# 3) estoque com memória: estoque[t] = lam * estoque[t-1] + fluxo[t]
def com_memoria(df, meia_vida=6):
    lam = 0.5 ** (1 / meia_vida)
    out = df.astype(float).copy()
    for i in range(1, len(df)):
        out.iloc[i] = lam * out.iloc[i - 1].values + df.iloc[i].values
    return out

nomes = dict(zip(info.Topic, info.Name))
top_ids = [t for t in top_ids if t in share.columns and t in nomes]

lam = 0.5 ** (1 / 8)
estoque = com_memoria(share[top_ids], meia_vida=6) * (1 - lam)

# 4) gráfico
nomes = info.set_index("Topic").Name
fig2 = go.Figure()
for tid in top_ids:
    fig2.add_trace(go.Scatter(x=estoque.index, y=estoque[tid],
                              mode="lines", name=nomes[tid]))
fig2.update_layout(title="Estoque de atenção (acumulada com memória)",
                   width=1000, height=500)
fig2.show()

In [44]:
topic_model.merge_topics(docs, [12, 14])

In [10]:
topic_model.get_topic_info(46)["Representation"].to_list()

[['tariffs',
  'tariff',
  'trading',
  'economic',
  'deals',
  'importers',
  'rate',
  'sectoral',
  'trade',
  '7pp']]

In [13]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.influencia_topicos import *

a = pd.DataFrame({
    "topic": topic_model.topics_,
    "date": pd.to_datetime(timestamps),
    "msg": df["original_message_id"]
})

pares_sob, grupos = sobreposicao(topic_model, limiar=0.80)
cont, share, clr  = preparar(a, grupos, freq="W")
topicos = maiores(cont, n=15)

subs = substituicao(clr, topicos)
ante = antecedencia(clr, topicos, max_lag=3)
coo  = coocorrencia(a, grupos)

print(subs)

[info] 87 períodos, 60 tópicos/grupos.
{'mais_negativos':      A   B  corr_diff  n_periodos
0    3  10  -0.285809          86
1    5   8  -0.276968          86
2    4  10  -0.267511          86
3   11  16  -0.253021          86
4    4  13  -0.231687          86
5   11  10  -0.212993          86
6    5   7  -0.211834          86
7    7  15  -0.200089          86
8    0   4  -0.194488          86
9    4  15  -0.187455          86
10   5  16  -0.186239          86
11   1  13  -0.179849          86
12   8   9  -0.155233          86
13   0   8  -0.149544          86
14   0  12  -0.143165          86
15  11   9  -0.130523          86
16  11   6  -0.123136          86
17   5  11  -0.108633          86
18   8  12  -0.102230          86
19   1  12  -0.097646          86, 'mais_positivos':      A   B  corr_diff  n_periodos
0    9  12   0.664419          86
1    7  16   0.437169          86
2   13  15   0.320043          86
3    5  15   0.284814          86
4    6  12   0.275343          86
5    

In [5]:
topic_model.reduce_topics(docs, nr_topics=70)